In [0]:
# Install packages for model comparison
%pip install optuna lightgbm catboost xgboost scikit-learn -q
print("✅ Packages installed")

# Restart Python kernel to load new packages
dbutils.library.restartPython()

In [0]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    classification_report, confusion_matrix, 
    precision_score, recall_score, f1_score, roc_auc_score
)
import warnings
warnings.filterwarnings('ignore')

# Add project to path
project_root = '/Workspace/Users/trannguyentoanphat1592005@gmail.com/mlops-e2e'
sys.path.append(project_root)

# Import feature engineering
from src.features.build_features import build_features

print("Imports successful")
print(f"Project root: {project_root}")

In [0]:
# Load relabeled data (with scientifically-based burnout labels)
data_path = os.path.join(project_root, 'data', 'raw', 'student_mental_health_burnout_relabeled.csv')
df = pd.read_csv(data_path)

print(f"Data loaded: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nTarget distribution:")
print(df['burnout_level'].value_counts(normalize=True))

# Check for missing values
print(f"\nMissing values: {df.isnull().sum().sum()}")
print(f"\nUsing relabeled dataset with feature-based burnout scores")

In [0]:
# Apply feature engineering
df = df.drop(columns=['student_id'])
df_enc = build_features(df, target_col='burnout_level')

# Convert boolean columns to int
for c in df_enc.select_dtypes(include=['bool']).columns:
    df_enc[c] = df_enc[c].astype(int)

print(f"Feature engineering complete: {df_enc.shape}")
print(f"Features: {df_enc.drop(columns=['burnout_level']).columns.tolist()}")

# Prepare X and y
X = df_enc.drop(columns=['burnout_level'])
y = df_enc['burnout_level']

print(f"\nX shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"\nClass distribution:")
print(y.value_counts())

In [0]:
# Stratified train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    stratify=y, 
    random_state=42
)

print(f"Train set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"\nTrain class distribution:")
print(y_train.value_counts())
print(f"\nTest class distribution:")
print(y_test.value_counts())

In [0]:
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import label_binarize

def evaluate_model(model, X_train, X_test, y_train, y_test, model_name="Model"):
    """Comprehensive model evaluation"""
    # Fit model
    model.fit(X_train, y_train)
    
    # Predictions
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)
    
    # Metrics
    precision = precision_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')
    
    # Multi-class ROC AUC (one-vs-rest)
    y_test_bin = label_binarize(y_test, classes=[0, 1, 2])
    roc_auc = roc_auc_score(y_test_bin, y_proba, average='weighted', multi_class='ovr')
    
    results = {
        'model': model_name,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'roc_auc': roc_auc
    }
    
    print(f"\n{'='*50}")
    print(f"{model_name} Results:")
    print(f"{'='*50}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1 Score:  {f1:.4f}")
    print(f"ROC AUC:   {roc_auc:.4f}")
    print(f"{'='*50}")
    
    return results, model, y_pred

print("Helper functions defined")

In [0]:
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
import time

results_list = []

print("Starting baseline model training...\n")

# 1. XGBoost Baseline (improved parameters)
print("1. Training XGBoost...")
start = time.time()
xgb_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=8,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='mlogloss',
    tree_method='hist'
)
xgb_results, xgb_fitted, xgb_pred = evaluate_model(
    xgb_model, X_train, X_test, y_train, y_test, "XGBoost (Baseline)"
)
xgb_results['train_time'] = time.time() - start
results_list.append(xgb_results)

# 2. Random Forest Baseline
print("\n2. Training Random Forest...")
start = time.time()
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf_results, rf_fitted, rf_pred = evaluate_model(
    rf_model, X_train, X_test, y_train, y_test, "Random Forest (Baseline)"
)
rf_results['train_time'] = time.time() - start
results_list.append(rf_results)

print("\nPart 1 complete (XGBoost & Random Forest)")

In [0]:
import lightgbm as lgb
from catboost import CatBoostClassifier
import time

# 3. LightGBM Baseline
print("3. Training LightGBM...")
start = time.time()
lgb_model = lgb.LGBMClassifier(
    n_estimators=200,
    max_depth=8,
    learning_rate=0.1,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    class_weight='balanced',
    random_state=42,
    verbose=-1
)
lgb_results, lgb_fitted, lgb_pred = evaluate_model(
    lgb_model, X_train, X_test, y_train, y_test, "LightGBM (Baseline)"
)
lgb_results['train_time'] = time.time() - start
results_list.append(lgb_results)

# 4. CatBoost Baseline
print("\n4. Training CatBoost...")
start = time.time()
cb_model = CatBoostClassifier(
    iterations=200,
    depth=8,
    learning_rate=0.1,
    auto_class_weights='Balanced',
    random_state=42,
    verbose=0
)
cb_results, cb_fitted, cb_pred = evaluate_model(
    cb_model, X_train, X_test, y_train, y_test, "CatBoost (Baseline)"
)
cb_results['train_time'] = time.time() - start
results_list.append(cb_results)

print("\nPart 2 complete (LightGBM & CatBoost)")

In [0]:
# Create comparison DataFrame
baseline_df = pd.DataFrame(results_list)
baseline_df = baseline_df.sort_values('roc_auc', ascending=False)

print("\n" + "="*80)
print("BASELINE MODELS COMPARISON")
print("="*80)
print(baseline_df.to_string(index=False))
print("="*80)

# Visualize results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: ROC AUC and F1 Score
ax1 = axes[0]
models = baseline_df['model']
x_pos = np.arange(len(models))
width = 0.35

ax1.bar(x_pos - width/2, baseline_df['roc_auc'], width, label='ROC AUC', alpha=0.8)
ax1.bar(x_pos + width/2, baseline_df['f1'], width, label='F1 Score', alpha=0.8)
ax1.set_xlabel('Model')
ax1.set_ylabel('Score')
ax1.set_title('Model Performance: ROC AUC vs F1 Score')
ax1.set_xticks(x_pos)
ax1.set_xticklabels(models, rotation=45, ha='right')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)
ax1.set_ylim([0.3, 1.0])

# Plot 2: Training Time
ax2 = axes[1]
ax2.bar(models, baseline_df['train_time'], alpha=0.8, color='coral')
ax2.set_xlabel('Model')
ax2.set_ylabel('Training Time (seconds)')
ax2.set_title('Training Time Comparison')
ax2.set_xticklabels(models, rotation=45, ha='right')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nBest baseline model: {baseline_df.iloc[0]['model']} (ROC AUC: {baseline_df.iloc[0]['roc_auc']:.4f})")
display(baseline_df)

In [0]:
# Check correlation between features and target
from scipy.stats import f_oneway, chi2_contingency

# Numeric features correlation with target
numeric_features = X_train.select_dtypes(include=[np.number]).columns.tolist()

print("Feature-Target Correlation (ANOVA F-statistic):")
print("="*60)
f_stats = {}
for feat in numeric_features:
    groups = [X_train[y_train == i][feat] for i in [0, 1, 2]]
    f_stat, p_value = f_oneway(*groups)
    f_stats[feat] = {'f_stat': f_stat, 'p_value': p_value}
    print(f"{feat:30s}: F={f_stat:8.2f}, p={p_value:.4f}")

# Sort by F-statistic
sorted_features = sorted(f_stats.items(), key=lambda x: x[1]['f_stat'], reverse=True)
print("\n" + "="*60)
print(f"Top 5 most discriminative features:")
for feat, stats in sorted_features[:5]:
    print(f"  {feat}: F={stats['f_stat']:.2f}")

# Check if any feature has significant correlation
significant = [f for f, s in f_stats.items() if s['p_value'] < 0.05]
print(f"\nFeatures with p<0.05: {len(significant)}/{len(numeric_features)}")

In [0]:
# Visualize feature distributions by class
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

# Select top 6 numeric features by F-statistic
top_features = [f[0] for f in sorted_features[:6]]

for idx, feat in enumerate(top_features):
    ax = axes[idx]
    for cls in [0, 1, 2]:
        mask = y_train == cls
        ax.hist(X_train[mask][feat], alpha=0.5, label=f'Class {cls}', bins=30)
    ax.set_title(feat)
    ax.set_xlabel('Value')
    ax.set_ylabel('Frequency')
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 If distributions overlap completely, features cannot distinguish classes!")

In [0]:
import optuna
from optuna.samplers import TPESampler

print("Starting Optuna hyperparameter tuning for LightGBM...")
print("This will run 25 trials to find optimal parameters.\n")

def objective_lgb(trial):
    """Optuna objective for LightGBM"""
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 15, 127),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'class_weight': 'balanced',
        'random_state': 42,
        'verbose': -1
    }
    
    model = lgb.LGBMClassifier(**params)
    model.fit(X_train, y_train)
    y_pred_proba = model.predict_proba(X_test)
    
    y_test_bin = label_binarize(y_test, classes=[0, 1, 2])
    roc_auc = roc_auc_score(y_test_bin, y_pred_proba, average='weighted', multi_class='ovr')
    
    return roc_auc

# Run Optuna study
optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
study.optimize(objective_lgb, n_trials=25, show_progress_bar=True)

print(f"\n{'='*60}")
print(f"Best ROC AUC: {study.best_value:.4f}")
print(f"Best parameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")
print(f"{'='*60}")

In [0]:
# Install onnxmltools if needed
try:
    import onnxmltools
except ImportError:
    %pip install onnxmltools onnxconverter-common -q
    import onnxmltools

# Train final LightGBM model with best parameters from Optuna
import lightgbm as lgb
import joblib
import json
from onnxmltools.convert import convert_lightgbm
from onnxmltools.convert.common.data_types import FloatTensorType

artifacts_dir = os.path.join(project_root, 'artifacts')
onnx_path = os.path.join(artifacts_dir, 'model.onnx')

print("Training final LightGBM model with best parameters...\n")

# Train model with best parameters from Optuna (LightGBM doesn't have auto_class_weights)
best_lgb = lgb.LGBMClassifier(
    **study.best_params,
    class_weight='balanced',  
    random_state=42,
    verbose=-1
)
best_lgb.fit(X_train, y_train)

print(f"Final model trained with best parameters")
print(f"   ROC AUC: {study.best_value:.4f}\n")

print("Exporting optimized LightGBM to ONNX...\n")

try:
    # Convert LightGBM to ONNX
    initial_type = [('float_input', FloatTensorType([None, X_train.shape[1]]))]
    onnx_model = convert_lightgbm(  
        best_lgb,
        initial_types=initial_type,
        target_opset=15
    )
    
    # Save ONNX model
    with open(onnx_path, 'wb') as f:
        f.write(onnx_model.SerializeToString())
    
    print(f"ONNX model exported successfully!")
    print(f"   Location: {onnx_path}")
    print(f"   Size: {os.path.getsize(onnx_path) / 1024 / 1024:.2f} MB")
    
except Exception as e:
    print(f"ONNX export failed: {e}")
    print(f"   Error details: {type(e).__name__}")
    print("\nFalling back to pickle format...")
    model_path = os.path.join(artifacts_dir, 'best_model.pkl')
    joblib.dump(best_lgb, model_path)
    print(f"Model saved as pickle: {model_path}")

# Also save feature columns metadata
feature_cols = X_train.columns.tolist()
with open(os.path.join(artifacts_dir, 'feature_columns.json'), 'w') as f:
    json.dump(feature_cols, f)

print(f"\nFeature columns saved: {len(feature_cols)} features")

In [0]:
import os
# Test the exported ONNX model
try:
    import onnxruntime as ort
except ImportError:
    %pip install onnxruntime -q
    import onnxruntime as ort

import numpy as np

print("Testing exported ONNX model...\n")

# Load ONNX model
onnx_path = os.path.join(project_root, 'artifacts', 'model.onnx')
sess = ort.InferenceSession(onnx_path)

# Get input/output names
input_name = sess.get_inputs()[0].name
output_names = [output.name for output in sess.get_outputs()]

print(f"ONNX model loaded successfully")
print(f"   Input name: {input_name}")
print(f"   Output names: {output_names}")
print(f"   Expected input shape: {sess.get_inputs()[0].shape}")

# Prepare test sample (ONNX expects 23 features, same as training)
test_sample = X_test.iloc[:5].values.astype(np.float32)
predictions = sess.run(output_names, {input_name: test_sample})

print(f"\nInference test successful!")
print(f"   Test sample shape: {test_sample.shape}")
print(f"   Predictions: {predictions[0][:5]}")
print(f"   True labels: {y_test.iloc[:5].values}")

# Compare with original LightGBM model
original_pred = best_lgb.predict(X_test.iloc[:5])
print(f"\nVerification:")
print(f"   ONNX predictions: {predictions[0][:5]}")
print(f"   Original LightGBM: {original_pred}")
match = np.array_equal(predictions[0][:5], original_pred)
print(f"   Match: {match}")

if match:
    print("\nSUCCESS: ONNX model produces identical predictions!")
else:
    print("\nMinor differences may occur due to numerical precision")